# WM-811K 입력 품질·OOD 기준 생성

선택이 끝난 CNN의 256차원 pooled feature를 사용해 학습 분포 기준을 만듭니다. train으로 클래스 기준을 적합하고 validation에서만 검토/OOD 임계값을 고정합니다. test는 기준 고정 후 평가에만 사용합니다.

T4 GPU 기준 약 5~15분을 예상합니다. 공간 셔플 stress는 파이프라인 점검용 합성 이상이며 실제 외부 장비 검증을 대체하지 않습니다.

## 1. T4 GPU와 Google Drive 연결

In [ ]:
!pip install -q joblib

from google.colab import drive, files
from pathlib import Path
import hashlib, json, shutil, subprocess, sys
import pandas as pd
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임 유형을 T4 GPU로 변경한 뒤 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))

## 2. 필요한 파일을 하나씩 업로드

아래 셀은 파일 선택 창을 세 번 엽니다. 다음 순서대로 정확히 하나씩 선택하세요.

1. `코드/wm811k/build_wm811k_ood.py`
2. `결과물/wm811k/selected_model_results/best_model.pt`
3. `결과물/wm811k/split_results/split_assignments.csv`

In [ ]:
def upload_exact(expected_name):
    print(f'{expected_name} 파일을 선택하세요.')
    uploaded = files.upload()
    assert len(uploaded) == 1, f'한 번에 파일 하나만 선택하세요: {list(uploaded)}'
    uploaded_name, raw_bytes = next(iter(uploaded.items()))
    expected_path, uploaded_path = Path(expected_name), Path(uploaded_name)
    renamed_copy = uploaded_path.stem.startswith(expected_path.stem + ' (')
    assert uploaded_path.suffix == expected_path.suffix and (uploaded_path.stem == expected_path.stem or renamed_copy), f'{expected_name} 대신 {uploaded_name}이 선택되었습니다.'
    destination = Path('/content') / expected_name
    destination.write_bytes(raw_bytes)
    print('업로드 완료:', destination, f'{destination.stat().st_size / 1024**2:.2f} MB')
    return destination

BUILD_SCRIPT = upload_exact('build_wm811k_ood.py')
CHECKPOINT_PATH = upload_exact('best_model.pt')
ASSIGNMENTS_PATH = upload_exact('split_assignments.csv')

## 3. Drive 전처리 배열 확인

In [ ]:
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_processed')
MAPS_PATH = DRIVE_DATA_DIR / 'wafer_maps_64.npy'
LABELS_PATH = DRIVE_DATA_DIR / 'labels.npy'
assert MAPS_PATH.is_file(), f'전처리 웨이퍼 배열이 없습니다: {MAPS_PATH}'
assert LABELS_PATH.is_file(), f'라벨 배열이 없습니다: {LABELS_PATH}'
assignments = pd.read_csv(ASSIGNMENTS_PATH)
assert set(assignments['split']) == {'train', 'validation', 'test'}
assert assignments['array_index'].is_unique
print(assignments.groupby('split').size())
print('maps:', MAPS_PATH)
print('labels:', LABELS_PATH)

## 4. GPU 특징 추출과 validation 임계값 고정

이 셀이 가장 오래 걸립니다. train 123,542개, validation 24,703개, test 24,705개와 validation 공간 셔플 stress 최대 5,000개를 순서대로 처리합니다.

In [ ]:
OUTPUT_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_ood/reference_run')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, '-u', str(BUILD_SCRIPT),
    '--maps', str(MAPS_PATH),
    '--labels', str(LABELS_PATH),
    '--assignments', str(ASSIGNMENTS_PATH),
    '--checkpoint', str(CHECKPOINT_PATH),
    '--output-dir', str(OUTPUT_DIR),
    '--device', 'cuda',
    '--batch-size', '512',
    '--num-workers', '2',
    '--review-quantile', '0.95',
    '--ood-quantile', '0.99',
    '--stress-samples', '5000',
    '--seed', '42',
]
completed = subprocess.run(command, check=False)
if completed.returncode != 0:
    raise RuntimeError(f'OOD 기준 생성 실패(exit={completed.returncode}). 위 로그의 첫 오류를 확인하세요.')
print('OOD 기준 생성 완료:', OUTPUT_DIR)

## 5. 결과 확인

In [ ]:
summary = json.loads((OUTPUT_DIR / 'ood_summary.json').read_text(encoding='utf-8'))
assert summary['threshold_selection']['calibration_split'] == 'validation'
assert summary['threshold_selection']['test_used_for_threshold'] is False
display(pd.DataFrame(summary['split_metrics']))
display(pd.DataFrame([summary['stress_metrics']]))
display(summary['threshold_selection'])
from IPython.display import Image, display as show
show(Image(filename=str(OUTPUT_DIR / 'ood_score_distribution.png')))

## 6. 결과 ZIP 다운로드

다운로드된 `wm811k_ood_results.zip`을 로컬 프로젝트의 `결과물/wm811k/colab_가져오기/` 폴더에 압축을 풀지 않고 넣으세요.

In [ ]:
required_outputs = {
    'ood_reference.json', 'ood_reference.npz', 'ood_summary.json', 'ood_split_metrics.csv',
    'ood_score_records.csv', 'ood_score_distribution.png', 'README.md',
}
missing = [name for name in required_outputs if not (OUTPUT_DIR / name).is_file()]
assert not missing, f'결과 파일 누락: {missing}'
ZIP_BASE = Path('/content/wm811k_ood_results')
ZIP_PATH = Path(shutil.make_archive(str(ZIP_BASE), 'zip', root_dir=OUTPUT_DIR))
zip_sha256 = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
print('ZIP:', ZIP_PATH, f'{ZIP_PATH.stat().st_size / 1024**2:.2f} MB')
print('SHA-256:', zip_sha256)
print('Drive 원본 결과:', OUTPUT_DIR)
files.download(str(ZIP_PATH))